In [0]:
from pyspark.sql.functions import current_timestamp

# 1. Day 2 Mock Data
new_data = [
    (1, 101, "250.00", "2023-10-01", "SHIPPED"), # UPDATE: Status changed to SHIPPED
    (4, 103, "120.00", "2023-10-03", "NEW")      # INSERT: Brand new order from a new customer
]
columns = ["order_id", "customer_id", "amount", "order_date", "status"]

# 2. Overwrite landing zone with Day 2 data
spark.createDataFrame(new_data, columns) \
     .write \
     .mode("overwrite") \
     .saveAsTable("deve.mock_data.landing_orders")

# 3. Read landing and append to Bronze
df_raw = spark.read.table("deve.mock_data.landing_orders")
df_bronze = df_raw.withColumn("ingest_time", current_timestamp())

df_bronze.write \
         .format("delta") \
         .mode("append") \
         .saveAsTable("deve.bronze.orders")

print("Day 2 data successfully appended to Bronze!")
display(spark.read.table("deve.bronze.orders").orderBy("order_id", "ingest_time"))